In [16]:
from heapq import heappush, heappop

# -------------------------
# Base de conocimiento (reglas)
# -------------------------
class KnowledgeBase:
    def __init__(self):
        self.rules = []

    def add_rule(self, rule):
        self.rules.append(rule)

    def apply_rules(self, edge, cost):
        """
        Aplica reglas al costo de una conexión
        """
        for rule in self.rules:
            cost = rule(edge, cost)
        return cost


In [17]:
# -------------------------
# Definición del grafo
# -------------------------
class TransportGraph:
    def __init__(self):
        self.graph = {}

    def add_edge(self, from_node, to_node, cost, line):
        self.graph.setdefault(from_node, []).append({
            "to": to_node,
            "cost": cost,
            "line": line
        })

    def neighbors(self, node):
        return self.graph.get(node, [])

In [ ]:
# -------------------------
# Algoritmo A*
# -------------------------
def heuristic(a, b):
    # Heurística simple (puede reemplazarse por una distancia real)
    return 0

def a_star(graph, kb, start, goal):
    queue = []
    # cada entrada: (prioridad, costo_actual, nodo, camino)
    heappush(queue, (0, 0, start, []))
    visited = set()

    while queue:
        priority, cost, node, path = heappop(queue)

        if node in visited:
            continue

        path = path + [node]
        visited.add(node)

        if node == goal:
            return path, cost

        for edge in graph.neighbors(node):
            edge_cost = kb.apply_rules(edge, edge["cost"])
            new_cost = cost + edge_cost
            new_priority = new_cost + heuristic(edge["to"], goal)
            heappush(queue, (new_priority, new_cost, edge["to"], path))
    return None, float("inf") 


In [19]:
# -------------------------
# Reglas inteligentes
# -------------------------

def penalizar_transbordo(edge, cost):
    # Si es una línea de transferencia, aumenta el costo
    if edge["line"] == "transfer":
        return cost + 5
    return cost

def evitar_linea_congestionada(edge, cost):
    # Penaliza líneas congestionadas (roja)
    if edge["line"] == "roja":
        return cost + 10
    return cost

def favorecer_linea_verde(edge, cost):
    # Si es línea verde, aplicar un sesgo favorable (reduce ligeramente el costo)
    if edge["line"] == "verde":
        return max(0, cost - 1)
    return cost

def penalizar_linea_construccion(edge, cost):
    # Si es línea amarilla (zonas de construcción), aumentar el costo
    if edge["line"] == "amarilla":
        return cost + 8
    return cost


In [20]:
# -------------------------
# Ejemplo de uso
# -------------------------
if __name__ == "__main__":

    # Crear grafo
    g = TransportGraph()
    # Nodos y aristas originales
    g.add_edge("A", "B", 5, "azul")
    g.add_edge("B", "C", 5, "azul")
    g.add_edge("A", "D", 3, "roja")
    g.add_edge("D", "C", 3, "roja")
    g.add_edge("B", "D", 2, "transfer")
    g.add_edge("C", "E", 4, "verde")
    g.add_edge("E", "F", 6, "verde")
    g.add_edge("D", "E", 2, "roja")
    g.add_edge("B", "F", 12, "amarilla")
    g.add_edge("F", "G", 3, "azul")

    # Base de conocimiento
    kb = KnowledgeBase()
    kb.add_rule(penalizar_transbordo)
    kb.add_rule(evitar_linea_congestionada)
    kb.add_rule(favorecer_linea_verde)
    kb.add_rule(penalizar_linea_construccion)

    # Buscar rutas de ejemplo
    ruta, costo = a_star(g, kb, "A", "C")
    ruta2, costo2 = a_star(g, kb, "A", "G")
    ruta1, costo1 = a_star(g, kb, "A", "D")

    print("Ruta óptima A->C:", ruta, "Costo:", costo)
    print("Ruta óptima A->D:", ruta1, "Costo:", costo1)
    print("Ruta óptima A->G:", ruta2, "Costo:", costo2)

Ruta óptima A->C: ['A', 'B', 'C'] Costo: 10
Ruta óptima A->D: ['A', 'B', 'D'] Costo: 12
Ruta óptima A->G: ['A', 'B', 'C', 'E', 'F', 'G'] Costo: 21
